# Data Vis: Plotting Time Series Data
* Notebook 5: Animating Time Series Data

## Setup

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
import seaborn as sns
from IPython.display import HTML


# Data

In this notebook, we will use a private dataset about (solar) power generation and use of a single family house in Germany. The dataset contains the following columns:
- `timestamp`: The date and time of the measurement. The data is recorded every 1 hour.
- `total_consumption_kwh`: The amount of power consumed per hour in kilowatts.
- `from_grid_kwh`: The amount of power provided from the grid per hour in kilowatts.
- `from_pv_kwh`: The amount of power generated by the solar panels per hour in kilowatts.
- `from_battery_kwh`: The amount of power provided by the battery per hour in kilowatts.
- `to_grid_kwh`: The amount of power provided to the grid per hour in kilowatts.
- `to_battery_kwh`: The amount of power provided to the battery per hour in kilowatts.
- `battery_percent`: The average percentage of battery charge at the time of measurement.
- `battery_kwh`: The average amount of power in the battery at the time of measurement in kilowatt hours.
- various weather data, including temperature, humidity, precipitation, wind speed, and solar radiation (ghi, dni, dhi).

In [ ]:
data = pd.read_csv("solar.csv")

Next, we extract several time-based features from the timestamp index. Breaking the timestamp into its individual components (year, month, day, hour, weekday) makes it straightforward to group and filter the data along those dimensions later. The `is_weekend` binary flag will be useful for separating weekday from weekend consumption patterns.

In [ ]:
data["timestamp"] = pd.to_datetime(data["timestamp"])

In [ ]:
data["year"] = data["timestamp"].dt.year
data["month"] = data["timestamp"].dt.month_name()
data["day"] = data["timestamp"].dt.day
data["hour"] = data["timestamp"].dt.hour
data["weekday"] = data["timestamp"].dt.day_name()
data["is_weekend"] = np.where(data["weekday"].isin(["Saturday", "Sunday"]), 1, 0)   

In [ ]:
data.set_index("timestamp", inplace=True)

In [ ]:
data.head()

# Animation

Animation is a powerful tool for visualizing time series data. It allows us to see how the data changes over time, making it easier to identify trends and patterns. In this notebook, we will create an animated plot of solar power generation over time.

Let's first start with a static plot of the data.

In [ ]:
data_1week = data["2024-07-01":"2024-07-07"]

plt.figure(figsize=(12, 6))
sns.lineplot(data=data_1week, x=data_1week.index, y="from_pv_kwh")
plt.title("Solar Power Generation Over Time")
plt.xlabel("Date")
plt.ylabel("Power Output (kWh)")
plt.show()

Now, let's turn this into an animated plot. We will use `Matplotlib`'s `FuncAnimation` to create the animation. Here is how it works:

1. **`fig` and `ax`**: The figure and axes are created *once* outside the animation loop. The `animate()` function updates this shared axes object on every frame.
2. **`animate(i)`**: Called once per frame, where `i` is the current frame index (0, 1, 2, …). On each call it clears the axes with `ax.clear()` and redraws a growing slice of the data (`data_1week.iloc[:i+1]`). This produces the progressive "drawing" effect as the line extends from left to right.
3. **Fixed axis limits**: We compute `xmin/xmax/ymin/ymax` from the full dataset *before* the animation starts and apply them inside `animate()`. This prevents the axes from auto-rescaling on every frame, which would cause a jarring jumping effect.
4. **`frames`**: The total number of frames to render — here, one frame per hourly data point.
5. **`interval`**: Delay in milliseconds between frames. A value of `100` gives 10 frames per second.
6. **`blit=True`**: An optimization that only redraws the parts of the figure that have actually changed, resulting in smoother playback. It requires `animate()` to return the list of updated Matplotlib *artists* (here, `ax.collections`).
7. **`plt.close(fig)`**: Closes the static Matplotlib figure so it does not render as a standalone image *in addition to* the animation widget below.
8. **`HTML(anim.to_jshtml())`**: Serializes the animation into a self-contained HTML/JavaScript player that runs interactively inside the notebook.

In [ ]:
xmin, xmax = data_1week.index.min(), data_1week.index.max()
ymin, ymax = data_1week["from_pv_kwh"].min(), data_1week["from_pv_kwh"].max()

fig, ax = plt.subplots(figsize=(12, 6))

def animate(i):
    ax.clear()                                                      # Wipe the previous frame's drawing
    subset = data_1week.iloc[: i + 1]                               # Cumulative slice: all data up to and including frame i
    sns.lineplot(x=subset.index, y=subset["from_pv_kwh"], ax=ax)    # Draw the line plot for the current subset of data
    ax.set_title(f"Frame {i}")                                      # Update the title to indicate the current frame number
    ax.set_xlim(xmin, xmax)                                         # Keep the x-axis stable across all frames
    ax.set_ylim(ymin, ymax)                                         # Keep the y-axis stable across all frames
    return ax.collections                                           # blit=True requires returning the updated axes elements

anim = FuncAnimation(
    fig,
    func=animate,
    frames=len(data_1week),   # One frame per hourly data point
    interval=100,             # 100 ms between frames → 10 fps
    blit=True,                # Only redraw changed artists for smoother performance
)

plt.close(fig)          # Prevent the static figure from rendering alongside the animation
HTML(anim.to_jshtml())  # Embed the animation as an interactive HTML/JS player


# Your Turn
**Try it yourself — some things to experiment with:**
- **Speed**: Decrease `interval` (e.g. `50`) to speed up the animation, or increase it (e.g. `300`) to slow it down.
- **Date range**: Change the slice `"2024-07-01":"2024-07-07"` to a full month to see seasonal patterns play out.
- **Column**: Replace `from_pv_kwh` with `total_consumption_kwh` or `battery_percent` to animate a different variable.
- **Title**: Inside `animate()`, replace `f"Frame {i}"` with `f"{subset.index[-1].strftime('%Y-%m-%d %H:%M')}"` to display the current timestamp instead of the frame number.

In [ ]:
# YOUR CODE HERE